In [ ]:
# pip install pandas numpy scikit-learn
# pip install cdt

In [49]:
import pandas as pd
import openpyxl
import numpy as np
import os

In [50]:
from sklearn.preprocessing import StandardScaler

In [51]:
from cdt.causality.pairwise import ANM, IGCI

In [52]:
FILE_PATH = r'C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Burnout\Copy of M1 Burnout Manuscript Data.xlsx'


In [53]:
# Causal Pair 1: Depression (X) -> Work-Related Burnout (Y)
X_COLUMN_NAME = 'Patient Health Questionnaire 9' 
Y_COLUMN_NAME = 'Work Related Burnout Score'

In [54]:
# --- Data Loading and Preprocessing ---

def load_and_prepare_data(filepath, x_col, y_col):
    """Loads data, selects the two variables, and performs standardization."""
    print(f"Loading data from: {filepath}")
    try:
        # Assuming the data is in the first sheet, adjust sheet_name if needed
        data = pd.read_excel(filepath)
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}. Please check your FILE_PATH.")
        return None, None
    except Exception as e:
        print(f"Error loading file: {e}")
        return None, None

    # Check if required columns exist
    if x_col not in data.columns or y_col not in data.columns:
        print(f"Error: One or both columns ('{x_col}', '{y_col}') not found in the dataset.")
        print("Available columns:", data.columns.tolist())
        return None, None

    # Select and clean the data
    df_pair = data[[x_col, y_col]].dropna()
    print(f"Data loaded. Remaining samples after cleaning: {len(df_pair)}")
    
    if len(df_pair) < 50: # Check for minimum size for reliable inference
         print("Warning: Dataset size is very small. Causal inference results may be unstable.")

    # Standardize data: Causal Discovery algorithms often work best on scaled data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_pair)
    
    # Extract standardized X and Y as NumPy arrays
    X_data = scaled_data[:, 0]
    Y_data = scaled_data[:, 1]

    return X_data, Y_data

In [55]:
# --- Causal Inference Functions ---

def run_anm(X, Y):
    """Runs the Additive Noise Model (ANM) algorithm."""
    print("\n--- Running Additive Noise Model (ANM) ---")
    
    # Initialize ANM (uses the default PyTorch backend)
    anm_model = ANM()

    # The predict method takes a tuple (X, Y) or (Y, X) and returns a score.
    # Score > 0 means X -> Y, Score < 0 means Y -> X.
    # The absolute value indicates confidence/strength.
    
    # CDT predicts a label: 1 (X->Y) or -1 (Y->X)
    # The 'anm_score' function provides the raw fitness score for X->Y
    score_x_to_y = anm_model.anm_score(X.reshape(-1, 1), Y.reshape(-1, 1))
    score_y_to_x = anm_model.anm_score(Y.reshape(-1, 1), X.reshape(-1, 1))

    # The preferred direction is the one with the higher fit score (closer to 1)
    if score_x_to_y < score_y_to_x:
        direction = f"{Y_COLUMN_NAME} -> {X_COLUMN_NAME}"
        confidence = score_y_to_x
    else:
        direction = f"{X_COLUMN_NAME} -> {Y_COLUMN_NAME}"
        confidence = score_x_to_y

    print(f"ANM Fit Score (X->Y): {score_x_to_y:.4f}")
    print(f"ANM Fit Score (Y->X): {score_y_to_x:.4f}")
    print(f"Inferred Causal Direction: {direction} (Confidence: {confidence:.4f})")
    
    # Note: In ANM, the true causal direction should have a better fit (lower independence test score, higher model score depending on implementation details). 
    # Here, we interpret the higher score as better fit.
    return direction


In [56]:
def run_igci(X, Y):
    """Runs the Information Geometric Causal Inference (IGCI) algorithm."""
    print("\n--- Running Information Geometric Causal Inference (IGCI) ---")
    
    # Initialize IGCI
    igci_model = IGCI()

    # IGCI's predict_proba returns a single score: 
    # Score > 0 indicates X -> Y
    # Score < 0 indicates Y -> X
    # The absolute magnitude indicates confidence/strength.
    
    # We pass the data tuple (X, Y)
    igci_score = igci_model.predict_proba((X, Y))

    if igci_score > 0:
        direction = f"{X_COLUMN_NAME} -> {Y_COLUMN_NAME}"
    else:
        direction = f"{Y_COLUMN_NAME} -> {X_COLUMN_NAME}"
        
    print(f"IGCI Score: {igci_score:.4f} (Positive = X->Y, Negative = Y->X)")
    print(f"Inferred Causal Direction: {direction} (Confidence: {abs(igci_score):.4f})")
    
    return direction


In [57]:
# --- Main Execution ---

if __name__ == "__main__":
    # 1. Load and Prepare Data
    X_data, Y_data = load_and_prepare_data(FILE_PATH, X_COLUMN_NAME, Y_COLUMN_NAME)

    if X_data is None:
        print("\nExiting due to data loading error. Please check configuration.")
    else:
        # 2. Run ANM Algorithm
        anm_result = run_anm(X_data, Y_data)

        # 3. Run IGCI Algorithm
        igci_result = run_igci(X_data, Y_data)
        
        print("\n--- Summary of Results ---")
        print(f"ANM Inferred Direction: {anm_result}")
        print(f"IGCI Inferred Direction: {igci_result}")

Loading data from: C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Burnout\Copy of M1 Burnout Manuscript Data.xlsx
Data loaded. Remaining samples after cleaning: 147

--- Running Additive Noise Model (ANM) ---
ANM Fit Score (X->Y): 6.3754
ANM Fit Score (Y->X): 6.5020
Inferred Causal Direction: Work Related Burnout Score -> Patient Health Questionnaire 9 (Confidence: 6.5020)

--- Running Information Geometric Causal Inference (IGCI) ---
IGCI Score: -0.0812 (Positive = X->Y, Negative = Y->X)
Inferred Causal Direction: Work Related Burnout Score -> Patient Health Questionnaire 9 (Confidence: 0.0812)

--- Summary of Results ---
ANM Inferred Direction: Work Related Burnout Score -> Patient Health Questionnaire 9
IGCI Inferred Direction: Work Related Burnout Score -> Patient Health Questionnaire 9


In [38]:
# Pair 2: children literacy

# --- Configuration ---
DATA_PATH = r"C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Literacy_children\DeterminantsDyslexia_Exp3_data_final.csv"
SHEET_NAME = 0 # Assuming the data is in the first sheet
SEPARATOR = ',' # Assuming European CSV format uses semicolon, common for Dutch datasets

# Column names required for the Composite Score (from your screenshot):
PHONO_SUBTESTS = [
    'T1_rhyme', 
    'T1_rhymeprime', 
    'T1_audsynt', 
    'T1_phondel'
]

In [42]:
# Variables for Causal Pair 2:
X_COLUMN_NAME = "Phonological_Awareness_Composite" 
Y_COLUMN_NAME = "T1_letter" # Letter Naming Test Score (T1)

# --- Data Loading and Cleaning ---
def load_and_preprocess_literacy_data(file_path, sheet_name):
    """Loads data, calculates the Phonological Awareness Composite score, and cleans data."""
    print(f"Loading data from: {file_path}")
    
    # 1. Load data (Updated to use pd.read_csv)
    try:
        # Assuming the CSV uses a comma separator (SEPARATOR variable)
        df = pd.read_csv(file_path, sep=SEPARATOR)
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}. Please check the path.")
        return None
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return None
    
    # 2. Check for missing columns
    if not all(col in df.columns for col in PHONO_SUBTESTS):
        missing_cols = [col for col in PHONO_SUBTESTS if col not in df.columns]
        print(f"Error: Missing required phonological subtest columns: {missing_cols}")
        print(f"Available columns: {df.columns.tolist()}")
        return None
        
    # 3. Calculate the Phonological Awareness Composite Score (SUM)
    # This assumes a simple summation of the subtests, as they are all T1 scores.
    # The thesis mentioned that these scores were transformed into Z-scores in the original study,
    # making summation the most appropriate way to create a combined measure.
    df[X_COLUMN_NAME] = df[PHONO_SUBTESTS].sum(axis=1)
    
    # --- VERIFICATION STEP ADDED ---
    print("\n--- Composite Score Verification ---")
    print(f"Descriptive Statistics for {X_COLUMN_NAME}:")
    print(df[X_COLUMN_NAME].describe())
    
    # 4. Filter for only the two required columns (plus potentially ID for debugging)
    required_cols = [X_COLUMN_NAME, Y_COLUMN_NAME]
    if Y_COLUMN_NAME not in df.columns:
        print(f"Error: Missing required effect column: {Y_COLUMN_NAME}")
        return None

    df_causal = df[required_cols].copy()
    
    # 5. Handle Missing Data (Imputation not recommended for Causal Discovery, so dropping NaNs)
    initial_samples = len(df_causal)
    df_causal.dropna(inplace=True)
    
    remaining_samples = len(df_causal)
    print(f"\nData loaded and Composite Score calculated.")
    print(f"Remaining samples after cleaning (used for Causal Pair 2): {remaining_samples} (Dropped {initial_samples - remaining_samples} rows with missing data)")
    
    return df_causal


In [45]:
# --- Causal Inference Functions ---

def run_anm(X, Y, x_name, y_name):
    """Runs the Additive Noise Model (ANM) algorithm."""
    print("\n--- Running Additive Noise Model (ANM) ---")
    anm_model = ANM()
    
    # Reshape data for ANM model
    X_arr = X.values.reshape(-1, 1)
    Y_arr = Y.values.reshape(-1, 1)
    
    # FIX: Calculate X->Y and Y->X scores separately to avoid IndexError on scalar return
    # The original return_all=True was not working as expected.
    
    # Score 1: X -> Y
    x_to_y_score = anm_model.predict_proba((X_arr, Y_arr))
    
    # Score 2: Y -> X (by swapping input variables)
    y_to_x_score = anm_model.predict_proba((Y_arr, X_arr))
    
    
    if x_to_y_score > y_to_x_score:
        inferred_direction = f"{x_name} -> {y_name}"
        confidence_score = x_to_y_score
    else:
        inferred_direction = f"{y_name} -> {x_name}"
        confidence_score = y_to_x_score

    print(f"ANM Causal Score (X->Y): {x_to_y_score:.4f}")
    print(f"ANM Causal Score (Y->X): {y_to_x_score:.4f}")
    print(f"Inferred Causal Direction (ANM): {inferred_direction} (Confidence: {confidence_score:.4f})")
    
    return inferred_direction, confidence_score

def run_igci(X, Y, x_name, y_name):
    """Runs the Information Geometric Causal Inference (IGCI) algorithm."""
    print("\n--- Running Information Geometric Causal Inference (IGCI) ---")
    
    igci_model = IGCI()

    # Convert pandas Series to numpy arrays for compatibility with IGCI's internal normalization logic
    X_arr = X.values 
    Y_arr = Y.values

    # IGCI's predict_proba returns a single signed score: 
    # Score > 0 indicates X -> Y
    # Score < 0 indicates Y -> X
    igci_score = igci_model.predict_proba((X_arr, Y_arr))

    if igci_score > 0:
        direction = f"{x_name} -> {y_name}"
    else:
        direction = f"{y_name} -> {x_name}"
        
    print(f"IGCI Score: {igci_score:.4f} (Positive = {x_name}->{y_name}, Negative = {y_name}->{x_name})")
    print(f"Inferred Causal Direction (IGCI): {direction} (Confidence: {abs(igci_score):.4f})")
    
    return direction, igci_score

In [48]:
# --- Main Execution ---
if __name__ == "__main__":
    
    df_causal = load_and_preprocess_literacy_data(DATA_PATH, SHEET_NAME)

    if df_causal is not None and not df_causal.empty:
        
        # Define X and Y based on the calculated composite score and the target variable
        X_data = df_causal[X_COLUMN_NAME]
        Y_data = df_causal[Y_COLUMN_NAME]

        # 1. Standardize data (crucial for ANM/IGCI)
        # Using StandardScaler to ensure X and Y are comparable, following best practices.
        scaler = StandardScaler()
        X_scaled = pd.Series(scaler.fit_transform(X_data.values.reshape(-1, 1)).flatten(), name=X_COLUMN_NAME)
        Y_scaled = pd.Series(scaler.fit_transform(Y_data.values.reshape(-1, 1)).flatten(), name=Y_COLUMN_NAME)

        print("\n--- Causal Pair 2 Analysis: Phonological Awareness <-> Letter Naming ---")
        
        # 2. Run ANM
        anm_direction, anm_score = run_anm(X_scaled, Y_scaled, X_COLUMN_NAME, Y_COLUMN_NAME)

        # 3. Run IGCI
        igci_direction, igci_score = run_igci(X_scaled, Y_scaled, X_COLUMN_NAME, Y_COLUMN_NAME)

        # 4. Summary
        print("\n--- Summary of Results ---")
        print(f"ANM Inferred Direction: {anm_direction}")
        print(f"IGCI Inferred Direction: {igci_direction}")

        # Check for consensus
        if anm_direction == igci_direction:
            print("\n!!! STRONG CONSENSUS: Both models agree on the causal direction. !!!")
        else:
            print("\n!!! DISAGREEMENT: Models conflict on the causal direction. !!!")
    else:
        print("\nCausal analysis skipped due to data loading/preprocessing errors.")

Loading data from: C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Literacy_children\DeterminantsDyslexia_Exp3_data_final.csv

--- Composite Score Verification ---
Descriptive Statistics for Phonological_Awareness_Composite:
count    205.000000
mean      56.102439
std       17.526010
min       13.000000
25%       44.000000
50%       55.000000
75%       68.000000
max      104.000000
Name: Phonological_Awareness_Composite, dtype: float64

Data loaded and Composite Score calculated.
Remaining samples after cleaning (used for Causal Pair 2): 205 (Dropped 0 rows with missing data)

--- Causal Pair 2 Analysis: Phonological Awareness <-> Letter Naming ---

--- Running Additive Noise Model (ANM) ---
ANM Causal Score (X->Y): 0.0000
ANM Causal Score (Y->X): -0.0000
Inferred Causal Direction (ANM): Phonological_Awareness_Composite -> T1_letter (Confidence: 0.0000)

--- Running Information Geometric Causal Inference (IGCI) ---
IGCI Score: -0.6988 (Positive = Phonological_Awareness_Composite->T1_lette

In [68]:
# Pair 3  ADHD Hyperactivity → Total Sleep Time

# Update this path to your DS9 TSV file:
HYPERACTIVITY_DATA_PATH = r"C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Sleep_ADHD\Hyperactivity&Impulsivity\31622-0009-Data.tsv" 
SLEEPING_DATA_PATH = r"C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Sleep_ADHD\Sleep\31622-0011-Data.tsv" 

# '\t' (tab) as the separator for TSV files
SEPARATOR = '\t' 

# Variables used for the Hyperactivity Composite Score (X)
ADHD_SUBTESTS = [
    'P6B47', # Youth can't sit still, is restless or hyperactive
    'P6B48'    # Youth is impulsive or acts without thinking
]
X_COLUMN_NAME = "Hyperactivity_Composite" 

# --- Function to Load and Calculate ---
def load_and_calculate_composite(file_path, subtests, composite_name, sep='\t'):
    """Loads a TSV file and calculates the composite score by summing specific columns."""
    
    print(f"Attempting to load data from: {file_path}")
    
    try:
        # Load data using the tab separator
        df = pd.read_csv(file_path, sep=sep)
        print(f"Data loaded successfully. Total rows: {len(df)}")
        
        # Check for missing required columns
        if not all(col in df.columns for col in subtests):
            missing_cols = [col for col in subtests if col not in df.columns]
            print(f"ERROR: Missing required columns in the file: {missing_cols}")
            return None
            
        # --- COMPUTE COMPOSITE SCORE BY SUMMING COLUMNS ---
        # axis=1 ensures the sum is done across the columns (row-wise), giving one total score per person.
        df[composite_name] = df[subtests].sum(axis=1)

        print("\n--- Composite Score Calculation Summary ---")
        print(f"Created new column: {composite_name}")
        print(f"Summed variables: {subtests}")
        print(f"Descriptive Statistics for {composite_name}:")
        print(df[composite_name].describe())
        
        return df
        
    except FileNotFoundError:
        print(f"ERROR: File not found at the specified path.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

# --- Execution ---
if __name__ == "__main__":
    
    # Load and process the hyperactivity data (DS9)
    df_adhd = load_and_calculate_composite(
        file_path=HYPERACTIVITY_DATA_PATH, 
        subtests=ADHD_SUBTESTS, 
        composite_name=X_COLUMN_NAME,
        sep=SEPARATOR
    )
    
    if df_adhd is not None:
        print("\nSuccessfully computed Hyperactivity Composite Score.")
    else:
        print("\nComposite score calculation failed.")

Attempting to load data from: C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Sleep_ADHD\Hyperactivity&Impulsivity\31622-0009-Data.tsv
Data loaded successfully. Total rows: 4898

--- Composite Score Calculation Summary ---
Created new column: Hyperactivity_Composite
Summed variables: ['P6B47', 'P6B48']
Descriptive Statistics for Hyperactivity_Composite:
count    4898.000000
mean       -2.898938
std         9.207642
min       -18.000000
25%       -18.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: Hyperactivity_Composite, dtype: float64

Successfully computed Hyperactivity Composite Score.


In [67]:
# Update this path to your DS9 TSV file:
HYPERACTIVITY_DATA_PATH = r"C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Sleep_ADHD\Hyperactivity&Impulsivity\31622-0009-Data.tsv" 

# Critical: Use '\t' (tab) as the separator for TSV files
SEPARATOR = '\t' 

# --- Function to Load and Print Columns ---
def load_and_print_columns(file_path, sep='\t'):
    """Loads a file and prints the list of all column names."""
    
    print(f"Attempting to load data from: {file_path}")
    
    try:
        # Load data using the tab separator
        df = pd.read_csv(file_path, sep=sep, encoding='utf-8')
        print(f"Data loaded successfully. Total rows: {len(df)}")
        
        # Extract and print the column names
        column_names = df.columns.tolist()
        
        print("\n--- EXACT COLUMN NAMES IN YOUR DATASET (Copy and Paste These) ---")
        for name in column_names:
            print(name)
            
        print("\nTotal number of columns:", len(column_names))
        
    except FileNotFoundError:
        print(f"\nERROR: File not found at the specified path: {file_path}")
    except Exception as e:
        print(f"\nAn unexpected error occurred during file loading: {e}")

# --- Execution ---
if __name__ == "__main__":
    load_and_print_columns(
        file_path=HYPERACTIVITY_DATA_PATH, 
        sep=SEPARATOR
    )

Attempting to load data from: C:\Users\Lenovo\Desktop\Licenta\Non_exp_Data\Sleep_ADHD\Hyperactivity&Impulsivity\31622-0009-Data.tsv
Data loaded successfully. Total rows: 4898

--- EXACT COLUMN NAMES IN YOUR DATASET (Copy and Paste These) ---
IDNUM
CP6SAMP
CP6PINT
CP6INTMON
CP6INTYR
CP6TELE
CP6SOURCE
CP6NATSM
CP6CITSM
CP6YLPCG
CP6YLOTH
CP6PCGREL
CP6SPAN
CP6AGE
CP6ADULT
CP6KIDS
CP6YAGEY
CP6YAGEM
CP6MRELF
CP6PRELB
CP6PMARB
CP6PCOHB
CP6PMARP
CP6PCOHP
CP6HHINC
CP6HHIMP
CP6HHSIZE
CP6POVCO
CP6POVCA
CP6EDU
CP6W9INTYR
CP6W9INTMON
CP6DROP
CP6CONF1
CP6CONF2
CP6CONF3
P6A1
P6A2_1
P6A2_2
P6A2_3
P6A2_4
P6A2_5
P6A2_91
P6A2_101
P6A3
P6A4
P6B1
P6B2
P6B3
P6B4
P6B5
P6B6
P6B7
P6B8
P6B9
P6B9_101
P6B9_102
P6B9_103
P6B9_104
P6B9_105
P6B9_106
P6B9_107
P6B10
P6B11
P6B12
P6B12_101
P6B12_102
P6B12_103
P6B12_104
P6B12_105
P6B13
P6B14
P6B15
P6B16
P6B17
P6B18
P6B19
P6B20
P6B21
P6B22
P6B23
P6B24
P6B25
P6B26
P6B27_1
P6B27_2
P6B27_3
P6B27_4
P6B27_5
P6B27_6
P6B27_7
P6B27_8
P6B27_9
P6B27_10
P6B27_11
P6B27_12
P6B28
P6B29
